In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

censo = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Sketch/Data_cleaning/Tabelas/Tabela_censo_Escolar_2024.csv", sep=';')
evasao = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Sketch/Data_cleaning/Tabelas/Tabela_Evasao_2024.csv")
enem = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Sketch/Data_cleaning/Tabelas/Tabela_ENEM_2024.csv")

# 2. Renomear as colunas para o padrão comum
censo = censo.rename(columns={"CO_ENTIDADE": "CO_ESCOLA"})
evasao = evasao.rename(columns={"codigo_escola": "CO_ESCOLA"})

df_completo = pd.merge(censo, evasao, on="CO_ESCOLA", how="inner")
df_completo = pd.merge(df_completo, enem, on="CO_ESCOLA", how="inner")

# Filtrar valores inconsistentes e nulos
df_completo = df_completo.dropna(subset=["MEDIA_INFRAESTRUTURA", "NOTA_GERAL", "evasao_medio_total"])
df_completo = df_completo[df_completo["NOTA_GERAL"] > 0]
df_completo = df_completo[
    (df_completo["evasao_medio_total"] >= 0)
    & (df_completo["evasao_medio_total"] <= 100)
]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_completo["MEDIA_INFRAESTRUTURA"],
        y=df_completo["NOTA_GERAL"],
        mode="markers",
        marker=dict(
            size=8 + (df_completo["evasao_medio_total"] / df_completo["evasao_medio_total"].max()) * 22
            if df_completo["evasao_medio_total"].max() > 0
            else 10,
            color=df_completo["evasao_medio_total"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Taxa de Evasão (%)"),
            line=dict(width=0.5, color="white"),
        ),
        text=[
            f"<b>{nome}</b><br>Infraestrutura: {infra:.2f}<br>Nota ENEM: {nota:.1f}<br>Evasão: {evasao:.1f}%"
            for nome, infra, nota, evasao in zip(
                df_completo["nome_escola"],
                df_completo["MEDIA_INFRAESTRUTURA"],
                df_completo["NOTA_GERAL"],
                df_completo["evasao_medio_total"],
            )
        ],
        hoverinfo="text",
    )
)

fig.update_layout(
    title=dict(
        text="Análise de Impacto: Infraestrutura vs Nota ENEM vs Evasão Escolar",
        x=0.5,
        font=dict(size=18),
    ),
    xaxis=dict(
        title="Média de Infraestrutura da Escola",
        gridcolor="lightgray",
        zeroline=False,
    ),
    yaxis=dict(
        title="Nota Geral do ENEM",
        gridcolor="lightgray",
        zeroline=False,
        range=[350, 850],
    ),
    plot_bgcolor="white",
    width=1100,
    height=650,
)

fig.show()